# Notebook 3: Promote and Decay

**Capture, Score, Gate, Promote, Decay.** The gate decided whether a memory gets in. These two steps decide what happens to it afterwards, and they are the part most memory systems skip. A store with a good gate and no lifecycle still fills up with session-level guesses that never get checked and never go away.

Promote: a memory earns its way up, from session to team to org, on corroboration from distinct sources. The top tier needs a human to sign off, by name. Decay: memories age. Each tier has a lifespan and a review interval, a session memory nobody read expires early, and org memories never expire on their own; they get flagged for a person to look at again.

This notebook is prebuilt. Nothing to write; read along while we run it. Every function takes `now`, so we will admit three memories, then move the clock to 4, 10, 100 and 200 days out and watch what happens to each. Decay touches every record in a store, so the whole notebook runs on a scratch copy of the handbook, seeded fresh here and deleted at the end. Your own store is read for a count and never written to.

In [1]:
# Make `import memory` work whether Jupyter opened the repo root or the notebooks folder.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "memory" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

with contextlib.redirect_stderr(io.StringIO()):  # a noisy third-party import warning we do not need in the room
    import weaviate.classes.init  # noqa: F401

from collections import Counter
from datetime import timedelta

from memory import Candidate, capture, collection_name, connect, describe_tier, fetch_all, score_candidate
from memory.lifecycle import (
    PROMOTION_RULES,
    SESSION_UNUSED_DAYS,
    TEAM_UNUSED_DAYS,
    corroborate,
    decay,
    endorse,
    promote,
    promote_all,
    promotion_decision,
    renew,
)
from memory.reference import reference_gate
from memory.schema import LIFESPAN_DAYS, REVIEW_DAYS, now_utc

print(describe_tier())

WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


In [2]:
client = connect()
if not client.collections.exists(collection_name()):
    raise RuntimeError("no collection yet. Run notebook 0 first; it creates and seeds the store.")
mine = client.collections.get(collection_name())  # your store: read for a count, never written to here
start_count = len(fetch_all(mine, status="active"))
if start_count == 0:
    raise RuntimeError("no active records. Run the 'fell behind' cell in notebook 0 to seed the handbook.")
print(f"your store holds {start_count} active records; this notebook will not change that")

from memory import ensure_collection
from memory.seed import seed_store

# A scratch copy of the handbook for the clock walk. Same 50 facts, same dates, separate collection,
# so decay() can run for real without touching anything you admitted in notebooks 1 and 2.
SCRATCH = collection_name("MemoryLifecycle")
collection = ensure_collection(client, name=SCRATCH, reset=True)
print(f"seeded {seed_store(collection)} handbook records into the scratch collection {SCRATCH}")

your store holds 50 active records; this notebook will not change that


seeded 50 handbook records into the scratch collection MemoryLifecycle


## The rules, on one screen

Three small tables in `memory/lifecycle.py` and `memory/schema.py` are the whole policy. Read them before the code; every result below traces back to one of these numbers.

In [3]:
print("PROMOTION_RULES   what it takes to move up into a tier")
for tier, rule in PROMOTION_RULES.items():
    print(
        f"  -> {tier:7s} {rule['min_corroboration']} distinct sources, confidence >= {rule['min_confidence']}, "
        f"{'human endorsement required' if rule['needs_endorsement'] else 'no endorsement needed'}"
    )
print("\nLIFESPAN_DAYS     how long a record lives in a tier before it expires (None: never on its own)")
print("REVIEW_DAYS       when a human is asked to look at it again")
for tier in LIFESPAN_DAYS:
    print(f"  {tier:8s} lifespan {str(LIFESPAN_DAYS[tier]):5s} review every {REVIEW_DAYS[tier]} days")
print(f"\nusage: a session memory unread for {SESSION_UNUSED_DAYS} days expires; a team memory idle for {TEAM_UNUSED_DAYS} days is flagged")

PROMOTION_RULES   what it takes to move up into a tier
  -> team    2 distinct sources, confidence >= 0.6, no endorsement needed
  -> org     3 distinct sources, confidence >= 0.7, human endorsement required

LIFESPAN_DAYS     how long a record lives in a tier before it expires (None: never on its own)
REVIEW_DAYS       when a human is asked to look at it again
  session  lifespan 7     review every 3 days
  team     lifespan 90    review every 30 days
  org      lifespan None  review every 180 days

usage: a session memory unread for 3 days expires; a team memory idle for 60 days is flagged


Two design choices are hiding in there. The org tier has no lifespan and the longest review interval, because that is where the whole company reads from and nothing there should vanish on a timer. And the org tier is the only one that needs an endorsement, because corroboration can be gamed by volume and a named human cannot.

## Three memories go in

Through the reference gate from notebook 2, at time zero. A customer preference an agent heard, a billing detail another agent heard, and a fiscal-year fact a person wrote in standup notes. All three start in the session tier, because every candidate does. Then the agent reads one of them back, with usage tracking on, so Decay will later know it was used.

In [4]:
from memory.evaluate import load_labeled
from memory.retrieve import recall, show

T0 = now_utc()


def at(days: int):
    """The simulated clock: T0 plus some days."""
    return T0 + timedelta(days=days)


labeled = {item["id"]: item for item in load_labeled()}
northgate = capture(
    text="Northgate Logistics asked for a 24 hour heads-up before freight deliveries.",
    source="agent:harbor", source_type="agent", context="support-session-9001",
    confidence=0.85, topic="customer", scope="customer:northgate-logistics",
)
halvorsen = Candidate.from_dict(labeled["good-01"])  # Halvorsen invoices, heard by agent:harbor
riverbend = Candidate.from_dict(labeled["good-02"])  # Riverbend fiscal year, from user:carla.mendes
demo = {"northgate": northgate, "halvorsen": halvorsen, "riverbend": riverbend}

gate = reference_gate()
uuids = {}
for name, cand in demo.items():
    cand.candidate_id = name
    d = gate.admit(collection, cand, score_candidate(collection, cand), now=T0)
    print(f"{name:10s}", d)
    assert d.outcome == "written", f"{name} was not written: {d.reason}"
    uuids[name] = d.record_uuid

print("\nthe agent reads the Riverbend fact back, usage tracked:")
# The query is a shipped text, so the cached vector is used and the embedding model is not needed here.
show(recall(collection, riverbend.text, scope="customer:riverbend-schools",
            tiers=["session"], limit=1, track_usage=True, now=T0))


def record(name):
    obj = collection.query.fetch_object_by_id(uuids[name])
    return {"uuid": str(obj.uuid), **obj.properties}


def board(title):
    print(f"\n{title}")
    for name in demo:
        r = record(name)
        print(f"  {name:10s} {r['tier']:8s} {r['status']:13s} x{r['corroboration_count']}  used {r['use_count']}  "
              f"review {r['review_at'].date()}  expires {r['expires_at'].date() if r['expires_at'].year < 2999 else 'never'}")


board("at T0:")
assert record("riverbend")["use_count"] == 1, "the tracked recall should have marked the Riverbend record as used"

northgate  ALLOW [written] 'Northgate Logistics asked for a 24 hour heads-up before freight delive'
halvorsen  ALLOW [written] 'Halvorsen Dental wants all invoices sent to accounts@halvorsendental.e'
riverbend  ALLOW [written] "Riverbend School District's fiscal year starts July 1, so their purcha"

the agent reads the Riverbend fact back, usage tracked:
1.000  [session] Riverbend School District's fiscal year starts July 1, so their purchase orders 
        from user:carla.mendes (human), confidence 0.90, x1

at T0:
  northgate  session  active        x1  used 0  review 2026-10-20  expires 2026-10-24
  halvorsen  session  active        x1  used 0  review 2026-10-20  expires 2026-10-24
  riverbend  session  active        x1  used 1  review 2026-10-20  expires 2026-10-24


Three session records, each vouched for by exactly one source (its own), each due for review in 3 days and gone in 7 unless something happens. The Riverbend record has been used once.

## Promote: a memory earns its way up

`promote()` applies the rules to one record and tells you what it did and why. Watch the Northgate preference climb, one step per call. Nothing here is automatic: a source has to corroborate, a person has to endorse, and the function says no until they do.

In [5]:
uid = uuids["northgate"]
print("1 source:              ", promote(collection, uid, now=T0))

corroborate(collection, uid, "user:ben.oyelaran", now=T0)
print("2 sources:             ", promote(collection, uid, now=at(1)))

r = corroborate(collection, uid, "agent:harbor", now=at(1))  # harbor already vouched; this does not count
print("harbor again:           corroboration_count still", r["corroboration_count"], "from", r["corroborated_by"])

corroborate(collection, uid, "user:carla.mendes", now=at(2))
print("3 sources, no endorser:", promote(collection, uid, now=at(2)))

endorse(collection, uid, "user:aisha.karim", now=at(3))
print("3 sources, endorsed:   ", promote(collection, uid, now=at(3)))

r = record("northgate")
print(f"\ntier {r['tier']}, endorsed by {r['endorsed_by']}, promoted {r['promoted_at'].date()}, "
      f"review {r['review_at'].date()}, expires {'never' if r['expires_at'].year == 2999 else r['expires_at'].date()}")

1 source:               (None, 'needs 2 distinct sources, has 1')
2 sources:              ('team', '2 sources, confidence 0.85')
harbor again:           corroboration_count still 2 from ['agent:harbor', 'user:ben.oyelaran']
3 sources, no endorser: (None, 'org tier needs a human endorsement')
3 sources, endorsed:    ('org', '3 sources, confidence 0.85, endorsed by user:aisha.karim')

tier org, endorsed by user:aisha.karim, promoted 2026-10-20, review 2027-04-18, expires never


Three things to read off that output. The same source vouching twice changes nothing; corroboration counts distinct sources, which is the lifecycle half of the `not_self_corroboration` policy you wrote in notebook 2. Three sources were not enough for the org tier; a person had to sign, and their name is on the record. And promotion reset the clock: the record entered org on day 3, so it never expires and comes due for review 180 days after that, not 180 days after capture.

`promotion_decision()` is the pure half of `promote()`: record in, next tier and reason out, no database. That is the function to unit test, and the one to point at when someone asks why a memory did not move.

In [6]:
for name in demo:
    print(f"{name:10s}", promotion_decision(record(name)))

print("\npromote_all at day 3, one pass over every active record:")
moved = [row for row in promote_all(collection, now=at(3)) if row["to"]]
print(f"  {len(moved)} promoted; the handbook is already at org and the other two have one source each")

northgate  (None, 'already at org')
halvorsen  (None, 'needs 2 distinct sources, has 1')
riverbend  (None, 'needs 2 distinct sources, has 1')

promote_all at day 3, one pass over every active record:
  0 promoted; the handbook is already at org and the other two have one source each


## Decay: memories age

`decay()` is one pass over every active record. For each one it picks a new status or leaves it alone, and it tells you why. It never deletes: expired and needs_review are statuses, and `recall()` only reads active records, so a decayed memory disappears from the agent's view while staying in the store for audit. Here is the clock walk.

In [7]:
def tick(days):
    actions = decay(collection, now=at(days))
    by_uuid = {a["uuid"]: a for a in actions}
    print(f"decay at day {days}: {dict(Counter(a['action'] for a in actions))}")
    for name in demo:
        a = by_uuid.get(uuids[name])
        print(f"  {name:10s} {a['action']:13s} {a['reason']}" if a else f"  {name:10s} (not active, skipped)")
    return actions


actions = tick(4)

decay at day 4: {'keep': 51, 'expired': 1, 'needs_review': 1}
  northgate  keep          healthy
  halvorsen  expired       never used in 4 days
  riverbend  needs_review  review was due 2026-10-20


Day 4. The Halvorsen record expired: nobody read it in 4 days, and the rule says 3. The Riverbend record was read, so it survived that rule, and instead got flagged: a session memory's review comes due at day 3. Northgate is in org and untouched, and so are the 50 handbook records. Note what did not happen: nothing was deleted. `recall()` reads active records only, so the expired record is out of the agent's view and still in the store for audit.

A flag is a question for a human, not an answer. Here is the person answering it. Carla confirms the fact still holds (`renew()`, which only accepts a needs_review record), two more sources turn out to agree, and the record is promoted rather than left to expire again next week.

In [8]:
uid = uuids["riverbend"]
r = renew(collection, uid, by="user:carla.mendes", now=at(4))
print("renewed:  ", r["status"], "review", r["review_at"].date(), "expires", r["expires_at"].date())

try:
    renew(collection, uuids["halvorsen"], by="user:carla.mendes", now=at(4))
except ValueError as e:
    print("halvorsen:", e)

corroborate(collection, uid, "agent:harbor", now=at(4))
corroborate(collection, uid, "user:ben.oyelaran", now=at(4))
print("promote:  ", promote(collection, uid, now=at(4)))
board("at day 4, after the human acted:")

renewed:   active review 2026-10-24 expires 2026-10-28
halvorsen: only needs_review records can be renewed; this one is expired
promote:   ('team', '3 sources, confidence 0.90')

at day 4, after the human acted:
  northgate  org      active        x3  used 0  review 2027-04-18  expires never
  halvorsen  session  expired       x1  used 0  review 2026-10-20  expires 2026-10-24
  riverbend  team     active        x3  used 1  review 2026-11-20  expires 2027-01-19


An expired record cannot be renewed; it was never flagged for a decision, it timed out, and bringing it back would be a new capture through the gate. The Riverbend fact is now a team memory with a 90-day lifespan and a 30-day review interval. Move the clock again.

In [9]:
actions = tick(10)
actions = tick(100)

decay at day 10: {'keep': 52}
  northgate  keep          healthy
  halvorsen  (not active, skipped)
  riverbend  keep          healthy
decay at day 100: {'keep': 51, 'expired': 1}
  northgate  keep          healthy
  halvorsen  (not active, skipped)
  riverbend  expired       team lifespan ended 2027-01-19


Day 10: everything holds. Day 100: the Riverbend team memory expired, because its 90-day lifespan started on day 4 and ran out on day 94. Had we stopped the clock at day 70, the idle rule would have flagged it first: last used on day 4, idle for 60 days, needs_review. We jumped past that, and the lifespan rule wins. Either way the signal is usage. If the fact had mattered, someone would have read it, `recall(track_usage=True)` would have moved `last_used`, and a human could have renewed or re-endorsed it. A memory nobody reads is not worth keeping at any tier below org.

One more jump.

In [10]:
actions = tick(200)

decay at day 200: {'needs_review': 51}
  northgate  needs_review  review was due 2027-04-18
  halvorsen  (not active, skipped)
  riverbend  (not active, skipped)


Day 200, and 51 records are flagged: Northgate, which entered org on day 3 and came due at day 183, and all 50 handbook facts, which were captured 30 days before they were seeded and came due at day 150. Nothing expired. That is the org tier working as designed: facts the whole company reads from never vanish on a timer, and they do not get to be true forever without anyone looking, either. A review queue of 51 is what a healthy store looks like at six months. Here is one of them being answered.

In [11]:
r = renew(collection, uuids["northgate"], by="user:aisha.karim", now=at(200))
print(f"northgate renewed by {r['endorsed_by']}: {r['status']}, next review {r['review_at'].date()}")
print("\nstore by status at day 200:", dict(Counter(x["status"] for x in fetch_all(collection))))

northgate renewed by user:aisha.karim: active, next review 2027-11-01

store by status at day 200: {'needs_review': 50, 'active': 1, 'expired': 2}


## Fell behind, or something broke?

Nothing in this notebook needs your code, so there is nothing to swap in. If the cells above stopped partway, run this one: it re-imports the reference gate and rebuilds the scratch copy from the handbook. Then run again from the "Three memories go in" cell, or carry on to the cleanup below.

In [12]:
from memory.reference import reference_gate  # noqa: F811

collection = ensure_collection(client, name=SCRATCH, reset=True)
print(f"rebuilt {SCRATCH} with {seed_store(collection)} handbook records; your own store was not touched")

rebuilt MemoryLifecycle with 50 handbook records; your own store was not touched


## Leave the store as you found it

The scratch copy goes. Your store was never written to, so there is nothing to put back. In production the same idea holds the other way round: because the lifecycle only ever changes `status`, a decay pass you regret is reversible one field at a time, and nothing has to be re-captured through the gate.

In [13]:
client.collections.delete(SCRATCH)
print(f"deleted {SCRATCH}")

deleted MemoryLifecycle


## What was Weaviate-specific here, and what was not

Weaviate-specific: `collection.data.update` for every status, tier and date change, `fetch_object_by_id`, and the scratch collection created and deleted around the demo. Any store with metadata updates has the same calls under other names.

Portable: the three rule tables, `promotion_decision()` and `decay_decision()` (pure functions over a record dict), the shape of corroborate, endorse, promote, decay and renew, the rule that the top tier needs a named human, and the rule that nothing is ever deleted by the lifecycle. The simulated clock is the testing trick to keep: every function takes `now`, so a year of aging is a unit test.

## Green check

In [14]:
count = len(fetch_all(mine, status="active"))
assert count == start_count, (
    f"your store started with {start_count} active records and now has {count}. This notebook never writes to "
    "it, so something else changed it; run the 'fell behind' cell in notebook 0 to reset it before notebook 4."
)
assert not client.collections.exists(SCRATCH), f"{SCRATCH} still exists; run the cleanup cell above"
print(f"\u2705 {describe_tier()}")
print(f"\u2705 {count} records in your store, untouched. Scratch copy deleted. Notebook 4 measures how well all of this works.")
client.close()

✅ WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord
✅ 50 records in your store, untouched. Scratch copy deleted. Notebook 4 measures how well all of this works.
